In [ ]:
import os
import json
import cv2
import numpy as np
from tqdm import tqdm

# ----------------------------
# Class mapping dictionary
# ----------------------------
cataracts_class_dict = {
    # Anatomy
    "Cornea": 0,
    "Iris": 1,
    "Pupil": 2,
    "Anterior capsule": 3,

    # Instruments
    "Charleux cannula": 4,
    "Rycroft cannula": 5,
    "Hydrodissection cannula": 6,
    "Viscoelastic cannula": 7,
    "Irrigation/aspiration handpiece": 8,
    "Phacoemulsifier handpiece": 9,
    "Lens injector": 10,
    "Capsule forceps": 11,
    "Capsulorhexis cystotome": 12,
    "Capsulorhexis forceps": 13,
    "Needle holder": 14,
    "Suture needle": 15,
    "Scissors": 16,
    "Vitrectomy handpiece": 17,
    "Lens loop": 18,
    "Cotton": 19,
    "Spatula": 20,
    "Filling cannula": 21,
    "Lens hook": 22,
    "Lens manipulator": 23,
    "Femto laser": 24,
    "Sponge": 25,
    "Vannas scissors": 26,
    "Implant injector": 27,
    "Irrigation handpiece": 28,
    "Primary incision knife": 29,
    "Secondary incision knife": 30,
    "Bonn forceps": 31,
    "Micromanipulator": 32,

    # Misc
    "Hand/finger": 33,
    "Eyelid speculum": 34,
    "Background": 35
}

# ----------------------------
# Path setup
# ----------------------------
root_dir = "/mnt/data/omkumar/mounted_datasets/Surgical/Cataract1K/Segmentation_dataset/Annotations/Images-and-Supervisely-Annotations"
output_root = "/mnt/data/omkumar/mounted_datasets/Surgical/Cataract1K/Segmentation_dataset/Masks"

os.makedirs(output_root, exist_ok=True)

# ----------------------------
# Loop through all cases
# ----------------------------
for case_name in tqdm(sorted(os.listdir(root_dir))):
    case_path = os.path.join(root_dir, case_name)
    ann_dir = os.path.join(case_path, "ann")
    if not os.path.isdir(ann_dir):
        continue  # skip non-case folders

    out_case_dir = os.path.join(output_root, case_name)
    os.makedirs(out_case_dir, exist_ok=True)

    for fname in os.listdir(ann_dir):
        if not fname.endswith(".json"):
            continue

        json_path = os.path.join(ann_dir, fname)
        with open(json_path) as f:
            data = json.load(f)

        height, width = data["size"]["height"], data["size"]["width"]
        mask = np.zeros((height, width), dtype=np.uint8)

        for obj in data["objects"]:
            cls_name = obj.get("classTitle")
            if cls_name not in cataracts_class_dict:
                print(f"⚠️ Unknown class '{cls_name}' in {json_path}")
                import sys
                sys.exit(0)

            class_id = cataracts_class_dict[cls_name]
            pts = np.array(obj["points"]["exterior"], np.int32)
            cv2.fillPoly(mask, [pts], color=class_id)

        # Save mask with matching name
        out_path = os.path.join(out_case_dir, fname.replace(".json", "_mask.png"))
        cv2.imwrite(out_path, mask)

print("✅ All masks saved successfully.")


True